# Return regression with topology (TDA) + risk with real tails

Improved version of `modelo_retorno_riesgo.ipynb` (kept as the baseline). Same data, same horizon (H = 21 days) and same walk-forward validation.

**What changes:**
1. **Saturated signal** `tanh(s)`: extreme momentum does not produce extreme forecasts (after a huge rally the trend does not keep going with the same strength).
2. **Topology (TDA):** for each asset, recent returns are turned into a point cloud (Takens embedding) and its **H0 persistent homology** is measured. It tells whether recent dynamics are *orderly* or *disorderly*; the model learns that the trend predicts better when they are orderly.
3. **Risk with real tails:** intervals, `P(loss)` and VaR come from the **empirical quantiles** of the errors, not from the normal (which underestimates extreme drops).

Final model: `z = a + b·tanh(s) + c·tanh(s)·TDA + d·TDA`, with `s` = average momentum over 1, 3, 6 and 12 months.

Everything with **NumPy + pandas** (rule in `model/README.md`: no ML libraries). Persistent homology is implemented by hand.

## 1. Setup and access to real data

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import erf, sqrt
warnings.filterwarnings("ignore")

H          = 21        # horizon in days (bars of each asset)
LAMBDA     = 0.97      # EWMA decay
TEST_FROM  = "2000-01-01"
TDA_WINDOW, TDA_DIM = 63, 3   # Takens: 63-day window embedded in R^3
SEED       = 42
rng = np.random.default_rng(SEED)

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fetch.py").exists()), None)
if ROOT is None:
    raise ImportError("Open this notebook inside the OxHack2026 repo (it needs data/fetch.py).")
sys.path.insert(0, str(ROOT))
from data.assets import all_symbols, label
from data.fetch import get_close, fetch_many, load
print("Repo:", ROOT)

In [ ]:
SYMBOLS = all_symbols()
missing = [s for s in SYMBOLS if load(s) is None]
if missing:
    print("Downloading (once):", missing)
    fetch_many(missing)

closes = {}
for s in SYMBOLS:
    try:
        c = get_close(s).dropna(); c = c[c > 0]
        if len(c) > 500:
            closes[s] = c
        else:
            print("  skipped (short history):", s)
    except Exception as e:
        print("  skipped", s, "->", type(e).__name__, e)
print(f"{len(closes)} assets with data")

## 2. Topology: persistent homology of recent dynamics

**Takens embedding.** The last 63 returns (scaled by their volatility) form a point cloud in R³:
each point is `(r_t, r_{t+1}, r_{t+2})`. The shape of that cloud summarises the dynamics: tightly packed points = regular moves;
scattered points with gaps = erratic moves.

**H0 persistent homology.** A ball grows around each point; when two balls touch, their components merge. The
"persistence" of each component is the radius at which it dies. For H0 those deaths are exactly the edges of the
**minimum spanning tree** (MST), computed with Prim's algorithm in NumPy.

**TDA feature:** the L2 norm of the persistence diagram, compared with its median over the last year (`log(norm / median)`).
Positive = more disorderly dynamics than usual; negative = more orderly.

In [ ]:
def ewma_vol(r, lam=LAMBDA):
    """Daily σ: σ²_t = λ·σ²_{t-1} + (1-λ)·r²_t, using data up to t."""
    return np.sqrt(r.pow(2).ewm(alpha=1-lam, adjust=False, min_periods=30).mean())

def h0_persistence(P):
    """Deaths in the H0 diagram of cloud P (Vietoris-Rips) = MST edges, via Prim."""
    D = np.sqrt(((P[:, None, :] - P[None, :, :])**2).sum(-1)); n = len(P)
    inside = np.zeros(n, bool); inside[0] = True
    dist = D[0].copy(); dist[0] = np.inf; deaths = np.empty(n-1)
    for k in range(n-1):
        j = np.argmin(dist); deaths[k] = dist[j]; inside[j] = True
        dist = np.minimum(dist, D[j]); dist[inside] = np.inf
    return deaths

def tda_takens(c, W=TDA_WINDOW, dim=TDA_DIM):
    """L2 norm of the H0 persistence of the Takens embedding, relative to its 252-day median."""
    r = np.log(c).diff().dropna()
    u = (r / ewma_vol(r)).dropna(); x = u.values
    out = np.full(len(x), np.nan)
    for t in range(W, len(x) + 1):
        win = x[t-W:t]
        P = np.column_stack([win[i:W-dim+1+i] for i in range(dim)])
        out[t-1] = np.sqrt((h0_persistence(P)**2).sum())
    s = pd.Series(out, index=u.index)
    return np.log(s / s.rolling(252, min_periods=126).median())

# Visual example: Takens cloud in an orderly vs. a disorderly period (SPY)
ex_asset = "SPY" if "SPY" in closes else list(closes)[0]
r_ex = np.log(closes[ex_asset]).diff().dropna(); u_ex = (r_ex / ewma_vol(r_ex)).dropna()
t_ex = tda_takens(closes[ex_asset]).dropna()
fig = plt.figure(figsize=(11, 4))
for i, (when, title) in enumerate([(t_ex.idxmin(), "most orderly"), (t_ex.idxmax(), "most disorderly")]):
    w = u_ex.loc[:when].values[-TDA_WINDOW:]; P = np.column_stack([w[k:TDA_WINDOW-TDA_DIM+1+k] for k in range(TDA_DIM)])
    ax = fig.add_subplot(1, 2, i+1, projection="3d"); ax.scatter(*P.T, s=8)
    ax.set_title(f"{label(ex_asset)} {when.date()}: {title} (TDA {t_ex[when]:+.2f})", fontsize=9)
plt.tight_layout(); plt.show()

## 3. Features, target and per-asset TDA
Target: `y = log(P_{t+H}/P_t)`, scaled `z = y / (σ·√H)`. Computing the TDA takes about a minute (one MST per day and asset).

In [ ]:
def build(s, c):
    r = np.log(c).diff().dropna()
    v = ewma_vol(r)
    f = pd.DataFrame(index=r.index)
    for k in (21, 63, 126, 252):
        f[f"mom{k}"] = r.rolling(k).sum() / (v*np.sqrt(k))
    f["tda"]   = tda_takens(c).reindex(r.index)
    f["y"]     = np.log(c.shift(-H) / c).reindex(r.index)
    f["sigma"] = v * np.sqrt(H)
    f["asset"] = s
    return f

TREND = ["mom21","mom63","mom126","mom252"]
panel = pd.concat([build(s, c) for s, c in closes.items()]); panel.index.name = "date"
panel[TREND] = panel[TREND].clip(-5, 5); panel["tda"] = panel["tda"].clip(-3, 3)
dataset = panel.dropna().reset_index().sort_values(["date","asset"]).reset_index(drop=True)
dataset["z"] = dataset["y"] / dataset["sigma"]
dataset["s"] = dataset[TREND].mean(1)
print(dataset.shape, dataset.date.min().date(), "→", dataset.date.max().date())

## 4. Time split (walk-forward with *purge*)
Test from 2000 in 6 blocks; each one is predicted with a model trained only on the past, with a `2H`-day gap.

In [ ]:
dates = np.sort(dataset.date.unique())
N_SPLITS, GAP = 6, pd.Timedelta(days=2*H+2)

def walk_forward():
    n = len(dates); start = np.searchsorted(dates, np.datetime64(TEST_FROM)); step = (n-start)//N_SPLITS
    for k in range(N_SPLITS):
        a = start + k*step; b = a + step if k < N_SPLITS-1 else n
        yield (dataset.date < dates[a] - GAP), (dataset.date >= dates[a]) & (dataset.date <= dates[b-1])

for i, (mtr, mte) in enumerate(walk_forward()):
    print(f"fold {i}: train until {dataset.date[mtr].max().date()} ({mtr.sum():,}) | "
          f"test {dataset.date[mte].min().date()}→{dataset.date[mte].max().date()} ({mte.sum():,})")

## 5. Models (least squares with different designs)
All are linear regressions on `z`; only the columns change:

| Model | Design |
|---|---|
| Historical mean | `1` |
| Trend (baseline model) | `1, s` |
| Saturated trend | `1, tanh(s)` |
| Trend + TDA | `1, s, s·TDA, TDA` |
| **Final: saturated + TDA** | `1, tanh(s), tanh(s)·TDA, TDA` |

In [ ]:
DESIGNS = {
    "Historical mean": lambda g: np.ones((len(g), 1)),
    "Trend":           lambda g: np.c_[np.ones(len(g)), g.s],
    "Saturated trend": lambda g: np.c_[np.ones(len(g)), np.tanh(g.s)],
    "Trend + TDA":     lambda g: np.c_[np.ones(len(g)), g.s, g.s*g.tda, g.tda],
    "Final: saturated + TDA": lambda g: np.c_[np.ones(len(g)), np.tanh(g.s), np.tanh(g.s)*g.tda, g.tda],
}
FINAL = "Final: saturated + TDA"

def fit(design, g):
    return np.linalg.lstsq(design(g), g.z.values, rcond=None)[0]

MODELS = ["Zero"] + list(DESIGNS)
zhat = {k: np.full(len(dataset), np.nan) for k in MODELS}
LEVELS = (0.90, 0.95, 0.99)
quant = {n: np.full((len(dataset), 2), np.nan) for n in LEVELS}   # empirical quantiles of the errors (in σ)
fold_residuals = {}
test_mask = np.zeros(len(dataset), bool)
for i, (mtr, mte) in enumerate(walk_forward()):
    tr, te = dataset[mtr], dataset[mte]
    zhat["Zero"][mte] = 0.0
    for k, des in DESIGNS.items():
        w = fit(des, tr); zhat[k][mte] = des(te) @ w
        if k == FINAL:
            res = tr.z.values - des(tr) @ w; fold_residuals[i] = (mte.values, res)
            for n in LEVELS:
                quant[n][mte] = np.quantile(res, [(1-n)/2, 1-(1-n)/2])
    test_mask |= mte.values

oos = dataset[test_mask].copy()
for k in MODELS:
    oos[k] = zhat[k][test_mask] * oos["sigma"]
print("Out-of-sample observations:", len(oos))

## 6. Return evaluation
- `R² vs mean`: improvement over the historical mean (the demanding test).
- `% years` / `% assets beating mean`: stability over time and across assets.

In [ ]:
def r2_vs(y, p, base):
    return 1 - np.sum((y - p)**2) / np.sum((y - base)**2)

def pct_beats(g, k, by):
    a = g.groupby(by).apply(lambda x: ((x.y - x[k])**2).sum() < ((x.y - x["Historical mean"])**2).sum())
    return a.mean()

y, base = oos.y.values, oos["Historical mean"].values
table = pd.DataFrame({k: {"MAE (pp)": np.mean(np.abs(y - oos[k]))*100,
                          "RMSE (pp)": np.sqrt(np.mean((y - oos[k])**2))*100,
                          "R² vs zero": r2_vs(y, oos[k].values, 0),
                          "R² vs mean": r2_vs(y, oos[k].values, base),
                          "% years beating mean": pct_beats(oos, k, oos.date.dt.year),
                          "% assets beating mean": pct_beats(oos, k, "asset")} for k in MODELS}).T
table.round(4)

### 6b. Is the improvement real? Bootstrap by year
Whole years are resampled (5,000 times) and each model's improvement over the **baseline Trend** is recomputed.
If the 90 % interval excludes 0, the improvement is robust; if it includes it, it is likely but not conclusive.

In [ ]:
years = oos.date.dt.year.values; U = np.unique(years)
sse = {k: np.array([((oos.y.values - oos[k].values)[years == t]**2).sum() for t in U]) for k in MODELS}
sse_m = sse["Historical mean"]
boot = {}
for k in ["Saturated trend", "Trend + TDA", FINAL]:
    d_ = sse["Trend"] - sse[k]; b = []
    for _ in range(5000):
        i = rng.integers(0, len(U), len(U)); b.append(d_[i].sum() / sse_m[i].sum())
    b = np.array(b)
    boot[k] = {"improvement vs Trend (pp of R²)": d_.sum()/sse_m.sum()*100,
               "CI90 low": np.percentile(b, 5)*100, "CI90 high": np.percentile(b, 95)*100,
               "P(improvement > 0)": (b > 0).mean()}
pd.DataFrame(boot).T.round(3)

### 6c. What the topology learns
Slope of the trend by TDA regime (terciles). If the TDA is useful, the trend should predict more in one regime than in another.

In [ ]:
terc = pd.qcut(dataset.tda, 3, labels=["orderly (low TDA)", "medium", "disorderly (high TDA)"])
reg = dataset.groupby(terc).apply(lambda g: pd.Series({
    "trend slope": np.cov(g.s, g.z)[0, 1] / g.s.var(),
    "mean return over H (%)": g.y.mean()*100, "cases": len(g)}))
display(reg.round(4))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
reg["trend slope"].plot.bar(ax=ax[0], rot=0); ax[0].set(title="How much does the trend predict?", ylabel="slope")
g = dataset[dataset.asset == ex_asset].set_index("date")
g.tda.rolling(21).mean().plot(ax=ax[1], lw=.8); ax[1].axhline(0, color="k", lw=.6)
ax[1].set(title=f"TDA of {label(ex_asset)} (21-day mean)", xlabel="")
plt.tight_layout(); plt.show()

## 7. Risk: EWMA σ + empirical quantiles
`σ` is still the EWMA volatility (λ = 0.97). What changes is **the shape** of the interval: instead of `mu ± 1.645σ` (normal),
it uses the quantiles of the historical errors `(y − mu)/σ`, computed on training data only.
This lets the interval be asymmetric (sharp drops are more frequent than sharp rises) and capture fat tails.

In [ ]:
ZN = {0.90: 1.6448536, 0.95: 1.9599640, 0.99: 2.5758293}
zres = (oos.y - oos[FINAL]).values / oos.sigma.values
cal = {}
for n in LEVELS:
    lo, hi = quant[n][test_mask].T
    cal[f"{n:.0%}"] = {"normal coverage": (np.abs(zres) <= ZN[n]).mean(),
                       "empirical coverage": ((zres >= lo) & (zres <= hi)).mean(),
                       "low quantile (σ)": lo.mean(), "high quantile (σ)": hi.mean(), "normal (σ)": ZN[n]}
calib = pd.DataFrame(cal).T
print(f"std of the errors in σ units: {zres.std():.3f} (ideal 1)")
calib.round(3)

In [ ]:
# Calibration of P(loss): when the model says 40 %, is money lost 40 % of the time?
p_norm = 0.5*(1 + np.vectorize(erf)((-oos[FINAL]/oos.sigma).values/sqrt(2)))
p_emp = np.full(len(oos), np.nan); threshold = (-oos[FINAL]/oos.sigma).values; idx_oos = np.where(test_mask)[0]
for i, (mte, res) in fold_residuals.items():
    sel = np.isin(idx_oos, np.where(mte)[0]); r_sorted = np.sort(res)
    p_emp[sel] = np.searchsorted(r_sorted, threshold[sel]) / len(r_sorted)
lost = (oos.y.values < 0).astype(float)
pd.DataFrame({"normal": {"mean P(loss)": p_norm.mean(), "actual frequency": lost.mean(), "Brier": np.mean((p_norm-lost)**2)},
              "empirical": {"mean P(loss)": p_emp.mean(), "actual frequency": lost.mean(), "Brier": np.mean((p_emp-lost)**2)}}).round(4)

## 8. Final model, fast `predict` and parameters for the API
Retrained on the **whole** history. `predict(prices, horizon_days)` returns the keys in `model/README.md`
(`rendimiento_esperado`, `volatilidad`, `prob_perdida`, `intervalo` at 90 %) plus `intervalo_95` and `var_95`.
For a horizon other than H, `mu` scales linearly and `σ` with √horizon.

**Fast:** last-day signals come from `model/predict_tda.signals_today`, which computes the TDA only for the 252 clouds
that enter the median (with a vectorised Prim) instead of the whole history. It gives exactly the same result as `build()`.

**For the API:** the coefficients and the error quantiles are saved to `model/params_tda.json`; `model/predict_tda.py`
reads them without retraining. That module follows the API conventions: `plazo_dias` in calendar days and simple returns.

In [ ]:
import json
from model.predict_tda import signals_today

W_FINAL = fit(DESIGNS[FINAL], dataset)
RESIDUALS = np.sort(dataset.z.values - DESIGNS[FINAL](dataset) @ W_FINAL)
print("Coefficients:", dict(zip(["a", "b·tanh(s)", "c·tanh(s)·TDA", "d·TDA"], W_FINAL.round(4))))

N_QUANTILES = 2000   # quantiles at the midpoints (i + 0.5)/N: enough for the CDF, intervals and mean
params = {"model": FINAL, "H": H, "lambda": LAMBDA, "tda_window": TDA_WINDOW, "tda_dim": TDA_DIM,
          "coef": W_FINAL.tolist(),
          "quantiles": np.quantile(RESIDUALS, (np.arange(N_QUANTILES) + 0.5) / N_QUANTILES).tolist(),
          "trained_until": str(dataset.date.max().date())}
(ROOT / "model" / "params_tda.json").write_text(json.dumps(params))
print("Parameters saved to model/params_tda.json")

def predict(prices: pd.Series, horizon_days: int = H) -> dict:
    """model/ contract: prices = daily closes with the date as index. Log returns, horizon in bars."""
    s, tda, sig_d = signals_today(prices, LAMBDA, TDA_WINDOW, TDA_DIM)
    row = pd.DataFrame({"s": [s], "tda": [tda]})
    z = float((DESIGNS[FINAL](row) @ W_FINAL)[0])
    sigma = sig_d * sqrt(horizon_days)
    mu = z * sig_d * sqrt(H) * (horizon_days / H)
    q = lambda p: float(np.quantile(RESIDUALS, p))
    return {"rendimiento_esperado": mu, "volatilidad": sigma,
            "prob_perdida": float(np.searchsorted(RESIDUALS, -mu/sigma) / len(RESIDUALS)),
            "intervalo": (mu + q(0.05)*sigma, mu + q(0.95)*sigma),
            "intervalo_95": (mu + q(0.025)*sigma, mu + q(0.975)*sigma),
            "var_95": -(mu + q(0.05)*sigma),
            "tda": tda}

In [ ]:
# Check: the fast path matches build() over the whole history, and how long each takes
import time
from model import predict_tda

rows = {}
for s_, c in closes.items():
    t0 = time.perf_counter(); f = build("x", c).iloc[-1]; t_slow = time.perf_counter() - t0
    s_slow = float(np.clip(f[TREND].astype(float), -5, 5).mean()); tda_slow = float(np.clip(f["tda"], -3, 3))
    t0 = time.perf_counter(); s_fast, tda_fast, _ = signals_today(c, LAMBDA, TDA_WINDOW, TDA_DIM); t_fast = time.perf_counter() - t0
    t0 = time.perf_counter(); api = predict_tda.predict(c, 30, es_cripto=s_.endswith("USDT")); t_api = time.perf_counter() - t0
    rows[label(s_)] = {"history (days)": len(c), "diff. s": abs(s_slow - s_fast), "diff. TDA": abs(tda_slow - tda_fast),
                       "slow (ms)": t_slow*1000, "fast (ms)": t_fast*1000, "API predict (ms)": t_api*1000,
                       "API: 30d return": api["rendimiento_esperado"], "API: P(loss)": api["prob_perdida"]}
check = pd.DataFrame(rows).T
assert check[["diff. s", "diff. TDA"]].max().max() < 1e-9, "the fast path does not match"
print(f"Identical for all {len(check)} assets. Mean time: slow {check['slow (ms)'].mean():.0f} ms → "
      f"fast {check['fast (ms)'].mean():.1f} ms ({check['slow (ms)'].mean()/check['fast (ms)'].mean():.0f}× faster)")
check.round(4)

## 9. Current forecast: "the model predicts a return of X %"

In [ ]:
rows = {}
for s, c in closes.items():
    p = predict(c, H)
    rows[s] = dict(asset=label(s), last_date=c.index[-1].date(), mu=p["rendimiento_esperado"], sigma=p["volatilidad"],
                   p_loss=p["prob_perdida"], ci90_low=p["intervalo"][0], ci90_high=p["intervalo"][1],
                   var_95=p["var_95"], tda=p["tda"])
forecast = pd.DataFrame(rows).T
for col in ["mu","sigma","p_loss","ci90_low","ci90_high","var_95","tda"]:
    forecast[col] = forecast[col].astype(float)
forecast = forecast.sort_values("mu", ascending=False)
for s, r in forecast.iterrows():
    print(f"{r.asset:<38} the model predicts a return of {r.mu*100:+.2f} %  (σ {r.sigma*100:.2f} %, P(loss) {r.p_loss:.0%}, "
          f"CI90 [{r.ci90_low*100:+.1f} %, {r.ci90_high*100:+.1f} %], TDA {r.tda:+.2f})")

## 10. Monte Carlo simulation with real tails (Gaussian copula)
Correlations come from a multivariate normal (last year), but each asset is mapped to the **empirical distribution**
of the errors: `R = mu + σ · empirical_quantile(Φ(N))`. This way the portfolio inherits the fat tails.

In [ ]:
N_SIM = 20000
rets = pd.DataFrame({s: np.log(c).diff() for s, c in closes.items()}).tail(365)[forecast.index]
corr = np.array(rets.corr(min_periods=60).fillna(0.0).values, dtype=float); np.fill_diagonal(corr, 1.0)
w_, V = np.linalg.eigh(corr); corr = (V * np.clip(w_, 1e-6, None)) @ V.T
dd = np.sqrt(np.diag(corr)); corr = corr / np.outer(dd, dd)

N = rng.multivariate_normal(np.zeros(len(corr)), corr, size=N_SIM)
U = 0.5*(1 + np.vectorize(erf)(N/sqrt(2)))
Zs = np.quantile(RESIDUALS, U)
sim = pd.DataFrame(forecast.mu.values + forecast.sigma.values * Zs, columns=forecast.index)
sim_n = pd.DataFrame(forecast.mu.values + forecast.sigma.values * N, columns=forecast.index)

portfolio, portfolio_n = sim.mean(1), sim_n.mean(1)
for name, pp in [("real tails", portfolio), ("normal", portfolio_n)]:
    print(f"Equal-weight portfolio over {H} days ({name:<10}): mean {pp.mean()*100:+.2f} % | VaR95 {-np.percentile(pp,5)*100:.2f} % | "
          f"VaR99 {-np.percentile(pp,1)*100:.2f} % | P(loss) {(pp<0).mean():.0%}")
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(portfolio_n*100, bins=80, alpha=.5, label="normal"); ax.hist(portfolio*100, bins=80, alpha=.5, label="real tails")
ax.axvline(0, color="k", lw=1); ax.set(title="Simulated portfolio distribution", xlabel="Return (%)"); ax.legend(); plt.show()

## 11. Export

In [ ]:
out = ROOT / "notebooks" / "outputs"; out.mkdir(parents=True, exist_ok=True)
forecast[["mu","sigma","p_loss","ci90_low","ci90_high","var_95"]].assign(horizon_days=H).to_csv(out / "pronostico_tda.csv")
table.to_csv(out / "metricas_tda_oos.csv"); calib.to_csv(out / "calibracion_tda.csv")
print("Saved to", out)

### Notes
- **What TDA means here:** only 0-dimensional homology (connected components) is used, which is equivalent to the MST and is computed in NumPy. Dimension 1 (loops) would need libraries such as `ripser` or `gudhi`.
- **What did not work** (tested outside this notebook): market-level TDA (joint cloud of 5 indices, Gidea-Katz style) as a return predictor, per-asset drift, and optimised weights per horizon. None beat the baseline model.
- **The improvement is likely, not guaranteed:** see the bootstrap (6b). With ~26 years of monthly data, differences of tenths of a point of R² cannot be confirmed with certainty.
- **The most solid part is the risk:** empirical quantiles fix the under-coverage of the normal at 95 % and 99 %, which is exactly where it matters for VaR.
- **Mixed calendars:** H counts bars of each asset (≈ 1 month for stocks, 3 weeks for crypto).